# 🧪 معمل التجارب: ربط واستخدام نماذج اللغات فائقة السرعة مع Groq API (Lecture 31)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في تطبيقات RAG الحقيقية، سرعة الاستجابة (Latency) هي العامل الأهم لتجربة المستخدم.
منصة **Groq** تقدم معالجات ثورية تسمى **LPUs (Language Processing Units)** قادرة على توليد أكثر من 300 كلمة في الثانية مجاناً!

في هذا المعمل ستتعلم:
1. الطريقة المباشرة للاستدعاء عبر `ChatGroq`.
2. الطريقة القياسية الموحدة `init_chat_model` المناسبة لمشاريع الشركات الكبيرة.
3. استعراض كافة الموديلات المتاحة والنشطة في حسابك.
4. البث المباشر للكلمات لحظة بلحظة (Streaming).
5. الدمج الكامل مع مسار الـ RAG وقاعدة ChromaDB.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. التحقق من مفتاح البيئة (`GROQ_API_KEY`) واستيراد المكتبات
**الفكرة ببساطة:**
بنفحص وجود مفتاح Groq السري في ملف `.env` ونتأكد من جاهزية الاتصال قبل إرسال أي طلبات.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print(f"✅ GROQ_API_KEY loaded successfully! (starts with: {groq_api_key[:8]}...)")
else:
    raise ValueError("❌ GROQ_API_KEY is missing! Please add it to your .env file.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب قراءة المفتاح والتحقق من وجوده بنفسك هنا):



### 2. الطريقة الأولى: الاستدعاء المباشر عبر `ChatGroq`
**الفكرة ببساطة:**
بنستخدم كلاس `ChatGroq` مباشرة ونحدد اسم الموديل والحرارة ونجرب نبعت له سؤال ونشوف سرعة الرد الخارقة.

In [ ]:
# method 1: direct provider instantiation via langchain-groq
from langchain_groq import ChatGroq

llm_direct = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512,
    api_key=groq_api_key
)

# test direct invocation
prompt_text = "Explain what makes Groq LPUs so fast in two concise sentences."
response = llm_direct.invoke(prompt_text)

print(f"❓ Prompt: {prompt_text}\n")
print(f"⚡ Response (via ChatGroq):\n{response.content}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استدعاء ChatGroq وسؤاله بنفسك هنا):



### 3. الطريقة الثانية: الواجهة القياسية الموحدة (`init_chat_model`)
**الفكرة ببساطة:**
في الشركات بنفضل `init_chat_model` لأنك تقدر تبدل الموديل من Groq لـ OpenAI أو Anthropic بمجرد تغيير سطر واحد، بدون ما تعدل بقية التطبيق نهائياً.

In [ ]:
# method 2: standard universal model initialization
from langchain.chat_models import init_chat_model

llm_unified = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.2,
    max_tokens=512
)

# test invocation with unified interface
prompt_unified = "In two concise sentences, what is the key difference between a CPU and a GPU?"
response_unified = llm_unified.invoke(prompt_unified)

print(f"❓ Prompt: {prompt_unified}\n")
print(f"⚡ Response (via init_chat_model):\n{response_unified.content}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استخدام init_chat_model بنفسك هنا):



### 4. استعراض النماذج المتاحة والنشطة على منصة Groq
**الفكرة ببساطة:**
بنتصل بـ API الرسمي لـ Groq ونجيب قايمة حية بأسماء الموديلات النشطة المتاحة لحسابك عشان تختار منها بسهولة.

In [ ]:
# list currently active models on Groq via official python SDK
import groq

client = groq.Groq(api_key=groq_api_key)
available_models = [model.id for model in client.models.list().data]

print(f"✅ Found {len(available_models)} active models on Groq:\n")
for idx, model_name in enumerate(sorted(available_models), start=1):
    print(f"  [{idx}] {model_name}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب سحب قائمة الموديلات المتاحة على Groq بنفسك هنا):



### 5. البث المباشر للكلمات لحظة بلحظة (Streaming)
**الفكرة ببساطة:**
بدل ما المستخدم يستنى الرد يكمل كله، بـ `.stream()` الكلمات بتظهر فورياً أول ما تخرج من المعالج زي شات GPT بالظبط.

In [ ]:
# test real-time streaming
stream_prompt = "Write 3 brief bullet points explaining why fast inference matters in AI chatbots."
print(f"❓ Prompt: {stream_prompt}\n")
print("🌊 Streaming Response: ")

for chunk in llm_direct.stream(stream_prompt):
    print(chunk.content, end="", flush=True)
print("\n\n✅ [Streaming Completed!]")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل الـ streaming بنفسك هنا):



### 6. الدمج الكامل مع مسار الـ RAG وقاعدة ChromaDB
**الفكرة ببساطة:**
بنربط موديل Groq مع قاعدة بيانات المتجهات المركزية بـ LCEL ونشوف إزاي بيجاوب على استعلام RAG في أجزاء من الثانية!

In [ ]:
# connect to centralized ChromaDB vector store
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# define context-grounded prompt
template = """You are a concise AI assistant. Use ONLY the following retrieved context to answer the question.
Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# construct lcel chain with fast Groq LLM
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_direct
    | StrOutputParser()
)

# execute test query
rag_query = "What is computer vision?"
rag_answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Groq RAG Answer:\n{rag_answer}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب دمج Groq مع مسار LCEL وقاعدة المتجهات بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 31):
1. **Groq LPU:** الخيار المثالي للتعلم والتطوير السريع بفضل سرعته الفائقة وتكلفته المجانية.
2. **`init_chat_model`:** المعيار الموحد في لانج تشين لفصل كود التطبيق عن مزود الموديل.
3. **التدفق (`.stream()`):** يمنح المستخدم تجربة تفاعلية فورية ومباشرة تشبه المنتجات التجارية الكبرى.